# Exact Parquet Symbol Search: Row-Group Pruning

**Status:** implemented and measured  
**Crate:** `spur-graph`  
**Production commit:** `0504903a42e62264b7d95d48e0b1c5a122276321`  
**Tracking:** `bd-3324` (implementation), `bd-3r2h` (this documentation)

## Executive conclusion

Exact symbol search now converts query equality constraints into a conservative Parquet metadata predicate before decoding rows. Row groups that cannot contain the requested symbol are skipped using exact min/max statistics and Bloom filters; candidate groups still pass through the existing Arrow row filter, sorting, and limit logic.

On the deterministic 131,072-symbol workload used for the change:

| Case | Before | After | Median-derived reduction | Median-derived speedup |
|---|---:|---:|---:|---:|
| Exact hit | 9.7851 ms | 2.1115 ms | 78.421% | 4.63× |
| Exact miss | 9.0533 ms | 14.976 µs | 99.835% | 604.52× |

Criterion’s distribution-level change estimates were −78.181% for the hit and −99.830% for the miss; these differ slightly from ratios computed from rounded median values.

The largest benefit is expected for missing or rare exact values in multi-row-group artifacts. Prefix and substring search semantics and execution paths are unchanged.

This notebook is the engineering guide for the optimization: what “alignment” means here, why the previous reader did extra work, how correctness is preserved, how the performance evidence was produced, and how to evolve the code safely.

## 1. What “data alignment” means in this change

The phrase can refer to three different concerns. Only the second is optimized here.

| Concern | Meaning | This change |
|---|---|---|
| Logical/schema alignment | Columns across batches have compatible names, types, and row counts | Preserved; no schema change |
| Physical row-group alignment | Rows are clustered into Parquet row groups whose metadata can rule predicates in or out | **Optimized through row-group and page pruning** |
| CPU/memory alignment | Arrow buffers begin at addresses suitable for vectorized/SIMD access | Unchanged |

The performance problem was therefore not misaligned Arrow buffers. It was a mismatch between an equality-shaped query and a scan path that did not use equality metadata to eliminate irrelevant physical row groups.

### Industry model

This follows the standard Parquet predicate-pushdown hierarchy:

1. Skip files or partitions when possible.
2. Skip row groups from footer statistics and Bloom filters.
3. Skip page ranges from page indexes.
4. Decode only predicate/projection columns.
5. Apply row filters during decoding.

Primary references:

- [Apache Parquet Bloom filters](https://parquet.apache.org/docs/file-format/bloomfilter/) describes a compact set over-approximation: a negative answer is definitive, while a positive answer means “possibly present.”
- [Arrow Rust predicate-pushdown guidance](https://arrow.apache.org/rust/parquet/arrow/arrow_reader/struct.ArrowReaderBuilder.html) distinguishes row-group pruning, page selection, projection, and row filtering.
- [Arrow Rust RowFilter](https://arrow.apache.org/rust/parquet/arrow/arrow_reader/struct.RowFilter.html) documents filtering during decode and late materialization.
- [Apache Arrow: Querying Parquet with Millisecond Latency](https://arrow.apache.org/blog/2022/12/26/querying-parquet-with-millisecond-latency/) explains why footer-driven row-group pruning avoids I/O and decode work.

**Key principle:** use cheap metadata to remove impossible work, but retain a row-level predicate as the final correctness gate.

## 2. Before and after

### Previous exact-search path

~~~mermaid
flowchart LR
    Q[Exact SearchOptions] --> O[Open nodes.parquet]
    O --> A[Construct Arrow RowFilter]
    A --> R[Reader visits every row group]
    R --> D[Decode predicate and projected columns]
    D --> F[Evaluate row_matches]
    F --> S[Sort candidates]
    S --> L[Apply result limit]
~~~

The Arrow row filter was useful: it evaluated predicate columns early and reduced later materialization. However, it did not by itself decide which row groups were impossible. The reader still visited every group.

A simplified cost model was:

~~~text
T_before ≈ metadata
         + Σ(read candidate columns for every row group)
         + Σ(decode/evaluate rows)
         + sort(matches)
~~~

### Current exact-search path

~~~mermaid
flowchart LR
    Q[Exact SearchOptions] --> P[Build conservative metadata predicate]
    P --> M[Read footer statistics and Bloom filters]
    M --> G{Can row group match?}
    G -- definitely no --> X[Skip group]
    G -- yes or uncertain --> I[Apply page selection]
    I --> D[Decode projected columns]
    D --> F[Apply original Arrow RowFilter]
    F --> S[Same sort and limit]
~~~

The corresponding cost becomes:

~~~text
T_after ≈ metadata
        + Σ(read/decode only groups that may match)
        + row_filter(candidate rows)
        + sort(matches)
~~~

Nothing about result ordering, result limits, or the final matching rules changed. The optimization removes provably irrelevant physical work before decoding.

## 3. Production implementation

The production entry point is:

- `crates/spur-graph/src/query_client.rs::ParquetClient::search_symbols_inner`
- Predicate builder: `exact_search_pruning_predicate`
- Shared reader: `crates/spur-graph/src/store/parquet.rs::read_filtered_projected_batches`

### Branch selection

`exact_search_pruning_predicate` returns `None` unless `SearchMode::Exact` is active. Prefix and substring searches therefore continue through the previous scan path.

For exact mode it builds:

~~~text
(entity_name == query OR qualified_name == query)
AND optional(file_path == filters.file)
AND optional(symbol_kind == filters.symbol_kind)
~~~

In Rust terms, the outer structure is `StringPruningPredicate::all(...)`; the two searchable name columns are combined with `StringPruningPredicate::any(...)`.

### Why each term is valid

- Exact search matches either the short entity name or fully qualified name, so those predicates must be joined with OR.
- An exact file filter and an exact symbol-kind filter are necessary conditions, so they can be joined with AND.
- A file glob is not converted into an equality predicate. It remains in the row filter because translating arbitrary glob semantics into metadata bounds would require a separately proven predicate family.

### Reader composition

The optimized branch calls `filtered_projected_batches` with:

- `SEARCH_COLUMNS` as the output projection;
- the new metadata pruning predicate;
- the existing `search_row_filter(schema, opts)` as the final row predicate.

Candidate batches are converted by the same `search_symbols_from_batch`, ordered by the same `compare_symbols`, and limited by the same `limited_search_result`.

This is an important review boundary: the change adds a pre-decode elimination layer; it does not replace the authoritative row matcher.

## 4. How pruning decides safely

The shared Parquet reader applies four complementary mechanisms.

| Layer | Implementation | Work avoided | Conservative fallback |
|---|---|---|---|
| Column projection | `with_projection` | Unrequested columns | Required output/predicate columns remain |
| Row-group pruning | `with_row_groups` | Entire impossible groups | Keep group if metadata is absent/uncertain |
| Page selection | `with_row_selection` | Impossible page ranges inside selected groups | Decode pages when index evidence is unavailable |
| Row filtering | `with_row_filter` | Nonmatching rows during decode | Final semantic predicate always runs |

For each equality leaf, `leaf_may_match_row_group` works in this order:

1. Resolve the Parquet column. If it is missing, return “may match” rather than skipping.
2. If min/max statistics are both exact and every requested value is outside the bounds, reject the row group.
3. Otherwise, if a column Bloom filter exists, retain the group only when at least one requested value is reported as possibly present.
4. If neither source proves absence, retain the group.

This policy intentionally prefers false positives over false negatives:

- **False positive:** an irrelevant group is decoded; performance is lower but results remain correct.
- **False negative:** a relevant group is skipped; results are wrong. The design and solver invariant forbid this.

Bloom-filter positives are never interpreted as proof that a row exists. They only mean the group must be checked by the exact row filter.

## 5. Correctness contract and solver proof

Define the authoritative row predicate:

~~~text
row_matches =
    (entity_name == query OR qualified_name == query)
    AND file_filter_matches
    AND symbol_kind_filter_matches
    AND file_glob_matches
~~~

Define metadata selection:

~~~text
row_group_selected =
    (entity_name_may_match OR qualified_name_may_match)
    AND file_path_may_match
    AND symbol_kind_may_match
~~~

The proof obligation is:

~~~text
For every row and query:
row_matches ⇒ row_group_selected
~~~

Equivalently, the solver searches for this counterexample:

~~~text
row_matches AND NOT row_group_selected
~~~

The result must be `UNSAT`.

### Pre/post solve evidence

| Check | Before implementation | After implementation |
|---|---|---|
| Model feasibility | SAT — `sol_3740b84960b24247` | SAT — `sol_87e8a7840ff544b9` |
| False-negative counterexample | UNSAT — `sol_fa5f18e867994d8d` | UNSAT — `sol_dfe69ec8bc9649fc` |

The implications supplied to the model state that an actual equality match must produce a metadata “may match” response for that column. This mirrors the implementation’s conservative behavior for missing columns, missing statistics, inexact bounds, and absent Bloom filters.

### Deliberate limitation

`file_glob_matches` is omitted from metadata selection. Omitting a condition from pruning can retain extra groups, but cannot remove a valid one. The authoritative row filter still evaluates the glob.

## 6. TDD proof: demonstrate a physical skip

Regression test:

`query_client::tests::exact_search_prunes_non_matching_row_groups`

The fixture creates two physical row groups:

1. A full first group containing one symbol named `target`.
2. A second group containing only nonmatching names.

After writing the artifact, the test locates the second group’s `entity_name` column chunk and overwrites that byte range with invalid data.

### RED observation

With the optimization branch disabled, exact search attempted to decode the irrelevant second group and failed while building/decoding the Arrow reader:

~~~text
Required field type_ is missing
~~~

### GREEN observation

With metadata pruning enabled, the second group is rejected before decode. Search succeeds and returns exactly one candidate whose `entity_name` is `target`.

This is stronger than a result-parity assertion alone. A parity test could pass even if all groups were still decoded. Corrupting an irrelevant group proves that the optimized reader physically avoids data that metadata excludes.

Additional semantic coverage comes from `tests/query_client_parity.rs`, which compares the Parquet and in-memory query clients.

## 7. Benchmark design

Benchmark:

`crates/spur-graph/benches/parquet.rs::bench_exact_search_row_group_pruning`

### Solved geometry

The benchmark geometry was selected with constraints rather than intuition:

- Fixed row-group size: 16,384 symbols.
- Allowed row-group count: 4 through 8.
- Objective: maximize row-group count within the bounded workload.
- Solved model: 8 groups × 16,384 rows = 131,072 symbols.
- Optimization result: `sol_e146ecc54cc34ff1`.
- Post-check of baked constants: SAT — `sol_e0db397d7ec74d77`.

The benchmark opens `nodes.parquet` metadata and asserts that the file contains exactly eight row groups. This prevents a writer/configuration change from silently turning the benchmark into a single-group scan.

### Data layout

- Every group has a distinct deterministic file path.
- The first row of the first group is named `target`.
- All other names are unique deterministic noise.
- Hit case: exact query `target`.
- Miss case: exact query `definitely_absent`.
- Result limit: 20.

The `ParquetClient` is opened before Criterion starts each timed iteration. Artifact construction, Parquet writing, metadata assertion, and client opening are setup costs—not part of the search latency.

### A/B protocol

The pre-pruning implementation and post-pruning implementation were run against the same benchmark code, data geometry, build profile, and machine. Criterion saved the pre-pruning sample as `pre_pruning`, then compared the optimized branch against that baseline.

In [3]:
# Recompute the benchmark interpretation from the recorded Criterion medians.
cases = [
    {"case": "exact hit", "before_ms": 9.7851, "after_ms": 2.1115},
    {"case": "exact miss", "before_ms": 9.0533, "after_ms": 0.014976},
]

row_group_rows = 16_384
row_group_count = 8
symbol_count = row_group_rows * row_group_count

assert symbol_count == 131_072
assert row_group_count > 1

print(f"Benchmark geometry: {row_group_count} × {row_group_rows:,} = {symbol_count:,} symbols")
print()
print("| case | before | after | latency reduction | speedup |")
print("|---|---:|---:|---:|---:|")
for case in cases:
    reduction = 100.0 * (1.0 - case["after_ms"] / case["before_ms"])
    speedup = case["before_ms"] / case["after_ms"]
    after = (
        f'{case["after_ms"] * 1000:.3f} µs'
        if case["after_ms"] < 1
        else f'{case["after_ms"]:.4f} ms'
    )
    print(
        f'| {case["case"]} | {case["before_ms"]:.4f} ms | {after} '
        f'| {reduction:.3f}% | {speedup:.2f}× |'
    )

Benchmark geometry: 8 × 16,384 = 131,072 symbols

| case | before | after | latency reduction | speedup |
|---|---:|---:|---:|---:|
| exact hit | 9.7851 ms | 2.1115 ms | 78.421% | 4.63× |
| exact miss | 9.0533 ms | 14.976 µs | 99.835% | 604.52× |


## 8. Results and production interpretation

| Case | Before | After | Median-derived reduction | Median-derived speedup |
|---|---:|---:|---:|---:|
| Exact hit | 9.7851 ms | 2.1115 ms | 78.421% | 4.63× |
| Exact miss | 9.0533 ms | 14.976 µs | 99.835% | 604.52× |

Criterion’s distribution-level estimates were −78.181% and −99.830%; the table above is recomputed from the rounded medians by the executable cell.

### Why the hit improves by about 4.6×

The target exists in the first group, so at least that group must still be decoded and checked. Metadata removes the other groups, but fixed query setup, metadata lookup, one-group decode, candidate conversion, sorting, and allocation remain.

An eight-group fixture does not imply an exact 8× speedup because total latency is not purely proportional to decoded row groups.

### Why the miss improves by about 604×

Every group can be rejected from metadata. The query performs metadata/Bloom-filter checks and returns without decoding 131,072 symbols. Exact misses are therefore the strongest case for this optimization.

### Expected workload sensitivity

Benefits grow when:

- the artifact contains multiple row groups;
- exact values are rare or absent;
- statistics/Bloom filters can exclude most groups;
- matching values are physically concentrated into few groups;
- storage or decompression cost is significant.

Benefits shrink when:

- the file contains one row group;
- a common value appears across most groups;
- metadata is absent or cannot prove exclusion;
- scan cost is dominated by fixed overhead;
- the query uses prefix or substring mode.

The benchmark demonstrates the optimization’s mechanism and upper-value cases. It is not a promise that every production query will achieve the same ratios.

## 9. Reproduction and measurement runbook

Run repository Rust commands through `scripts/spur-cargo`; do not invoke bare `cargo`.

### Correctness and compilation

~~~bash
scripts/spur-cargo test -p spur-graph \
  query_client::tests::exact_search_prunes_non_matching_row_groups \
  -- --exact --nocapture

scripts/spur-cargo test -p spur-graph --test query_client_parity

scripts/spur-cargo check -p spur-graph --benches

scripts/spur-cargo fmt --all -- --check
~~~

### Benchmark smoke

This executes each benchmark case once and verifies the eight-row-group assertion:

~~~bash
SPUR_REMOTE=0 scripts/spur-cargo bench \
  -p spur-graph \
  --bench parquet \
  bench_exact_search_row_group_pruning \
  -- --test
~~~

### Criterion A/B comparison

On the unoptimized implementation, save a baseline:

~~~bash
SPUR_REMOTE=0 scripts/spur-cargo bench \
  -p spur-graph \
  --bench parquet \
  bench_exact_search_row_group_pruning \
  -- --save-baseline pre_pruning
~~~

On the optimized implementation, compare against it:

~~~bash
SPUR_REMOTE=0 scripts/spur-cargo bench \
  -p spur-graph \
  --bench parquet \
  bench_exact_search_row_group_pruning \
  -- --baseline pre_pruning
~~~

Use an isolated worktree or reversible patch when switching implementations. Keep the machine, power mode, compiler flags, benchmark geometry, and target directory stable. Do not compare samples taken with different row-group counts or query cases.

### Fixture behavior

The general Parquet benchmark loader accepts `SPUR_GRAPH_PERF_FIXTURE`. An explicit invalid path now fails clearly. When no explicit override is provided and the machine-local baseline fixture is absent, the benchmark derives the repository root from `CARGO_MANIFEST_DIR` and builds current facts. The deterministic pruning benchmark does not depend on the machine-local fixture.

## 10. Maintenance guidelines

### When changing search semantics

1. Treat `row_matches` as the authoritative semantics.
2. Add metadata pruning only for conditions that are necessary for every true row match.
3. Preserve the same AND/OR structure:
   - entity name OR qualified name;
   - exact file and symbol kind joined with AND.
4. Keep unsupported or hard-to-prove predicates in the row filter.
5. Re-run the false-negative model after changing predicate composition.
6. Add parity cases for every newly pruned filter.

### When adding a new filter

A filter may enter `exact_search_pruning_predicate` only if all are true:

- its row-level semantics are precisely defined;
- Parquet metadata can represent a conservative “may match” test;
- missing/inexact metadata retains the group;
- a false-negative counterexample remains UNSAT;
- TDD demonstrates that a provably irrelevant corrupt group is skipped;
- parity tests show unchanged results.

Do not add file-glob, regex, fuzzy, prefix, or substring pruning by approximating them as equality.

### When changing Parquet writing

- Keep the benchmark row-group assertion.
- Confirm Bloom filters and exact statistics remain available for searchable string columns.
- Re-run hit and miss cases.
- Measure file-size/write-time cost as well as read latency if Bloom-filter settings change.
- Treat row-group size as a workload trade-off: smaller groups improve pruning granularity but increase metadata and scheduling overhead.

### Regression signals

Investigate if:

- the miss case rises from microseconds back toward full-scan milliseconds;
- hit and miss converge unexpectedly;
- the row-group assertion fails;
- the corrupt-group TDD test starts decoding the irrelevant group;
- Parquet/in-memory parity differs;
- benchmark setup dominates because client creation moved into the timed closure.

### Non-goal

Do not use this notebook as justification for changing Arrow allocator alignment, SIMD boundaries, or buffer padding. Those require a separate profile-driven investigation.

## 11. Evidence ledger and review checklist

### Implementation evidence

- Commit: `0504903a42e62264b7d95d48e0b1c5a122276321`
- Commit message: `feat(spur-graph): bd-3324 prune exact parquet searches`
- Production file: `crates/spur-graph/src/query_client.rs`
- Benchmark file: `crates/spur-graph/benches/parquet.rs`
- Diff size: 211 insertions, 21 deletions across exactly those two files.

### Verification recorded during implementation

- TDD RED: irrelevant corrupt group was decoded and produced an Arrow reader failure.
- TDD GREEN: exact search skipped that group and returned the target.
- Query-client unit subset: 13 passed.
- Query-client parity integration: 7 passed.
- Benchmark compilation: passed.
- Hit/miss benchmark smoke: passed.
- Scoped Clippy, formatting, and diff checks: passed.
- Independent review: no Critical or Important findings.
- Two Minor findings were addressed:
  - assert the benchmark’s actual row-group count;
  - reject an invalid explicit fixture override rather than silently falling back.

A full `spur-graph` test attempt also reported seven unrelated extractor golden-snapshot mismatches. No extractor or golden files were changed by this optimization; focused query-client verification remained green.

### Reviewer checklist

Before approving a future modification, verify:

- [ ] Exact results and ordering remain identical to the in-memory client.
- [ ] Every metadata predicate is necessary for a true row match.
- [ ] Unknown metadata means “retain,” never “skip.”
- [ ] File-glob and non-exact modes have not been unsafely promoted.
- [ ] The corrupt-row-group regression still proves a physical skip.
- [ ] The benchmark still creates and asserts eight row groups.
- [ ] Hit and miss Criterion samples use identical workload geometry.
- [ ] Performance claims state workload limits instead of extrapolating universally.

---

**Decision:** retain the optimization. It follows Parquet’s standard metadata-pruning model, has a proven no-false-negative contract, and provides a large measured reduction in exact-search latency on selective multi-row-group workloads.